In [ ]:
# Ghost cell to set font style for the presentation
from IPython.display import HTML

HTML("""
<style>
    /* Scales code inside the input cells (where you type) */
    .jp-CodeMirror, .cm-content {
        font-size: 18px !important;
    }
    
    /* Scales plain text output (e.g., print statements) */
    .jp-OutputArea-output pre {
        font-size: 16px !important;
    }

    /* Scales markdown cell prose */
    .jp-MarkdownOutput, .jp-RenderedHTMLCommon {
        font-size: 20px !important; 
        line-height: 1.5 !important;
    }
    
    /* Scales markdown headers specifically */
    .jp-RenderedHTMLCommon h1 { font-size: 2.2em !important; }
    .jp-RenderedHTMLCommon h2 { font-size: 1.8em !important; }
    .jp-RenderedHTMLCommon h3 { font-size: 1.5em !important; }
</style>
""")

<div style="background-color: #141b29; padding: 20px; border-radius: 8px; border: 1px solid #233047; color: #e0e0e0;">

# 🌊 Graphflood in `pytopotoolbox`

`graphflood` <a href="https://esurf.copernicus.org/articles/12/1295/2024/">(Gailleton et al., 2024)</a> simulates how water moves across terrain surfaces using a fast 2D stationary diffusive wave approximation.

Give it an elevation map (DEM) and some water (rainfall fields or point discharge paths), and it routes flow until it outputs balanced, equilibrated 2D parameter maps:

<table style="width: 100%; border-collapse: collapse; margin: 20px 0; font-size: 22px;">
    <thead>
        <tr style="background-color: rgba(58, 134, 255, 0.2); border-bottom: 2px solid #3a86ff;">
            <th style="padding: 12px; text-align: left; width: 70%;">Dynamic Metric</th>
            <th style="padding: 12px; text-align: left; width: 30%;">Variable</th>
        </tr>
    </thead>
    <tbody>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 15px;">🔹 <strong>Flow Depth</strong></td>
            <td style="padding: 15px;">$h$</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 15px;">🔹 <strong>Velocity</strong></td>
            <td style="padding: 15px;">$u$</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 15px;">🔹 <strong>Shear Stress</strong></td>
            <td style="padding: 15px;">$\tau$</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 15px;">🔹 <strong>Discharge</strong></td>
            <td style="padding: 15px;">$Q$</td>
        </tr>
    </tbody>
</table>

---

### 📦 Prerequisites

Install dependencies via your terminal:
```bash
pip install topotoolbox
pip install ipympl  # Optional: For interactive figures
```


### Offline assistance?

- [Github Discussion](https://github.com/orgs/TopoToolbox/discussions)
- Last resort: grop me an email (boris.gailleton@univ-rennes.fr)
</div>

In [ ]:
# General imports
import topotoolbox as ttb
import matplotlib.pyplot as plt
import numpy as np
import time
import scipy.stats as st
from scipy.ndimage import binary_closing, distance_transform_edt
from skimage.morphology import skeletonize

# Set matplotlib backend
# %matplotlib inline
## If you want interactive plots
%matplotlib ipympl


In [ ]:
# --- Webinar & High-Contrast Readable Settings ---
plt.rcParams['figure.figsize'] = (10, 6)   # Large default size for screen sharing
plt.rcParams['figure.dpi'] = 110          # Crisp rendering on high-res streams
plt.rcParams['font.size'] = 14            # Large text for readability
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

# --- Monospace Font Strategy ---
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['font.monospace'] = ['Fira Code', 'JetBrains Mono', 'Consolas', 'Courier New', 'monospace']

# --- Cyberpunk / Geek Dark Mode Theme ---
plt.rcParams['figure.facecolor'] = '#121214'    # Deep charcoal background
plt.rcParams['axes.facecolor'] = '#1a1a1e'      # Slightly lighter plot area
plt.rcParams['axes.edgecolor'] = '#444444'      # Subtle borders
plt.rcParams['axes.linewidth'] = 1.5

# High-contrast text colors
plt.rcParams['text.color'] = '#e0e0e0'
plt.rcParams['axes.labelcolor'] = '#e0e0e0'
plt.rcParams['xtick.color'] = '#a0a0a0'
plt.rcParams['ytick.color'] = '#a0a0a0'

# Grid settings
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#2d2d34'
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.linewidth'] = 0.8
plt.rcParams['grid.alpha'] = 0.4

# --- Neon / Synthwave Color Palette ---
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=[
    '#00ffcc', '#ff007f', '#9d4edd', '#3a86ff', '#ffb703', '#70e000'
])

# Image defaults
plt.rcParams['image.cmap'] = 'magma'  # Dark-mode default colormap


<div style="background-color: #121824; padding: 20px; border-radius: 8px; border-left: 6px solid #00ffcc; color: #e0e0e0;">

# ⚡ Quickstart

Provide a raw DEM and your water inputs to calculate balanced flow conditions and track fluid behavior maps instantly.

</div>

In [ ]:
# Load DEM and crop to a specific spatial subset via percentage thresholds
dem = ttb.load_dem('greenriver')
dem = dem.crop(0.35, 0.68, 0.3, 0.85, 'percent')

# if you want to load your DEM:
# dem = ttb.read_tif('mydem.tif')

# Plot the cropped topography with an overlaid hillshade
fig, ax = plt.subplots()
cb = ax.imshow(dem.z, cmap='terrain', extent=dem.extent)
ax.imshow(dem.hillshade(), cmap='gray', extent=dem.extent, alpha=0.6)
plt.colorbar(cb, label='elevation (m)')
plt.show()

<div style="background-color: #121824; padding: 20px; border-radius: 8px; border-left: 6px solid #9d4edd; color: #e0e0e0;">

## 🛠️ `GFObject` (Graphflood Object)

The primary interface utilized to initialize and run the `graphflood` engine.

<table style="width: 100%; border-collapse: collapse; margin: 15px 0; font-size: 20px;">
    <thead>
        <tr style="background-color: rgba(157, 78, 221, 0.2); border-bottom: 2px solid #9d4edd;">
            <th style="padding: 10px; text-align: left; width: 25%;">Argument</th>
            <th style="padding: 10px; text-align: left; width: 15%;">Requirement</th>
            <th style="padding: 10px; text-align: left; width: 60%;">Description</th>
        </tr>
    </thead>
    <tbody>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1); background-color: rgba(255, 255, 255, 0.02);">
            <td style="padding: 12px;">❗ <strong>grid</strong></td>
            <td style="padding: 12px;"><code style="color: #ff007f;">Required</code></td>
            <td style="padding: 12px;">Core topography/DEM input grid layout.</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;">⚙️ <strong>bcs</strong></td>
            <td style="padding: 12px;"><code>Optional</code></td>
            <td style="padding: 12px;">Pixel boundary condition codes. Defaults to unconstrained borders.</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;">⚙️ <strong>p</strong></td>
            <td style="padding: 12px;"><code>Optional</code></td>
            <td style="padding: 12px;">Precipitation supply rates (scalar value or 2D array matrix in $m/s$).</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;">⚙️ <strong>manning</strong></td>
            <td style="padding: 12px;"><code>Optional</code></td>
            <td style="padding: 12px;">Manning surface roughness index ($n$) used during flow calculations.</td>
        </tr>
    </tbody>
</table>

</div>

In [ ]:
# Instantiate the Graphflood object with 100 mm/h uniform precipitation converted to m/s
## 100 mm/h is super high, I chose it to make sure even in our v. small demo site we generate enough flooding
gfo = ttb.GFObject(dem, p=(100e-3)/3600, manning=0.05)

<div style="background-color: #121824; padding: 20px; border-radius: 8px; border-left: 6px solid #00ffcc; color: #e0e0e0;">

## 🔄 Running the Model

The solver routes surface water incrementally across cells until it settles into equilibrium.

<table style="width: 100%; border-collapse: collapse; margin: 15px 0; font-size: 20px;">
    <thead>
        <tr style="background-color: rgba(0, 255, 204, 0.15); border-bottom: 2px solid #00ffcc;">
            <th style="padding: 10px; text-align: left; width: 25%;">Control Parameter</th>
            <th style="padding: 10px; text-align: left; width: 45%;">Impact on Convergence</th>
            <th style="padding: 10px; text-align: left; width: 30%;">Constraint / Risk</th>
        </tr>
    </thead>
    <tbody>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;">🔢 <strong>Iterations</strong></td>
            <td style="padding: 12px;">Higher execution runs bring the solver closer to true mathematical flow alignment.</td>
            <td style="padding: 12px; color: #a0a0a0;">Increases computational compute time.</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;">⏱️ <strong>Time Step</strong> (<code>dt</code>)</td>
            <td style="padding: 12px;">Larger steps speed up convergence rates towards a steady-state layout.</td>
            <td style="padding: 12px; color: #ffb703;"><strong>Excessive values trigger numerical instability and cause solution degradation.</strong></td>
        </tr>
    </tbody>
</table>

<div style="padding: 10px 15px; background-color: rgba(255, 183, 3, 0.08); border-left: 4px solid #ffb703; border-radius: 4px; font-size: 18px;">
    ⚠️ <strong>Key Distinction:</strong> The parameter <code>dt</code> functions purely as a mathematical tuning knob to speed up calculations. It does not reflect actual physical timeline tracking.
</div>

</div>

In [ ]:
# Run the steady-state solver loop for 100 fixed steps
gfo.run_n_iterations(dt=1e-2, n_iterations=100)

In [ ]:
# Display calculated water depths layered over the relief hillshade
fig, ax = plt.subplots()
cb = ax.imshow(gfo.hw, cmap='Blues', extent=dem.extent, vmax=0.5)
ax.imshow(dem.hillshade(), cmap='gray', extent=dem.extent, alpha=0.4)
plt.colorbar(cb, label='Flow depth (m)')
plt.show()

<div style="background-color: #121824; padding: 20px; border-radius: 8px; border-left: 6px solid #ff007f; color: #e0e0e0;">

## 📊 Computed Metrics

### 💥 Shear Stress ($\tau$)

The total dragging friction force exerted by shifting water directly along the surface channel bed.

$$\tau = \rho g h S_w$$

<div style="padding: 10px 15px; background-color: rgba(255, 0, 127, 0.08); border-left: 4px solid #ff007f; border-radius: 4px; font-size: 18px;">
    ⚙️ <strong>Engine Execution:</strong> Extracted directly using <code>gfo.compute_tau()</code> based on calculated local depth ($h$) and hydraulic water surface slope profiles ($S_w$).
</div>

</div>

In [ ]:
# Compute and map boundary shear stress across the channel bed
shear_stress = gfo.compute_tau()

# 
fig, ax = plt.subplots()
cb = ax.imshow(shear_stress, cmap='magma', extent=dem.extent, vmax=100)
ax.imshow(dem.hillshade(), cmap='gray', extent=dem.extent, alpha=0.4)
plt.colorbar(cb, label=r'Shear Stress ($\tau$)')
plt.show()


<div style="background-color: #121824; padding: 20px; border-radius: 8px; border-left: 6px solid #3a86ff; color: #e0e0e0;">

### 🐆 Flow Velocity

Calculates flow velocity magnitudes ($u$) across your target grid landscape layout.

<div style="padding: 10px 15px; background-color: rgba(58, 134, 255, 0.08); border-left: 4px solid #3a86ff; border-radius: 4px; font-size: 22px;">
    ⚙️ <strong>Execution:</strong> <code>gfo.get_u()</code>
    <br>🚀 <strong>Roadmap:</strong> Directional vector component tracking layouts will follow in an upcoming update.
</div>

</div>

In [ ]:
# Retrieve depth-averaged flow velocity magnitudes
flow_velocity = gfo.get_u()

fig, ax = plt.subplots()
cb = ax.imshow(flow_velocity, cmap='viridis', extent=dem.extent, vmax = 2.)
ax.imshow(dem.hillshade(), cmap='gray', extent=dem.extent, alpha=0.4)
plt.colorbar(cb, label=r'$u$ flow velocity $\frac{m}{s}$')
plt.show()

<div style="background-color: #121824; padding: 20px; border-radius: 8px; border-left: 6px solid #00ffcc; color: #e0e0e0;">

### 📉 Discharge & Hydraulic Slope

A custom hydraulic twist on classical geomorphic **slope-area plotting**, replacing upstream drainage basin surface area ($A$) with explicitly calculated flow discharge tracking values ($Q$).

<div style="padding: 10px 15px; background-color: rgba(0, 255, 204, 0.08); border-left: 4px solid #00ffcc; border-radius: 4px; font-size: 20px;">
    🔍 <strong>Core Objective:</strong> Compares log-scaled surface slope records ($S_w$) against discharge rates ($Q$) to distinguish distinct flow channels and landscape scaling domains.
</div>

</div>

In [ ]:
# Flatten volumetric discharge and hydraulic water surface slope grids into log space
Q_in = np.log10(gfo.get_qvol_i().z.ravel())
Sw = np.log10(gfo.get_sw().z.ravel())

# Filter out NaN and infinite elements
mask = np.isfinite(Q_in) & np.isfinite(Sw)
Q_in = Q_in[mask]
Sw = Sw[mask]

# 1. Compute 2D Density (Binned Counts)
bins2D = 60
stat_2d, x_edges, y_edges, _ = st.binned_statistic_2d(Q_in, Sw, None, 'count', bins=bins2D)
# 
# 2. Compute 1D Profile (Median of Y binned by X)
bins = 30
bin_means, bin_edges, _ = st.binned_statistic(Q_in, Sw, statistic='median', bins=bins)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# 3. Plot
fig, ax = plt.subplots()

# 2D Density Map distribution
ax.imshow(stat_2d.T, origin='lower', cmap='magma', aspect='auto',
               extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]])

# 1D Overlaid Median Line trend line
ax.scatter(bin_centers, bin_means, facecolor='cyan', edgecolor='gray', lw=3, s=200)

ax.set_ylim(-2, -0.5)
ax.set_xlabel('log Q')
ax.set_ylabel('log Sw')
plt.show()


In [ ]:
# Initialize empty mask grid matching the topography geometry
domain = np.zeros_like(dem.z)
# 
text_arrays = """
Extract raw arrays for discharge and water surface slopes
"""
Qi2d = np.log10(gfo.get_qvol_i().z)
Sw2d = np.log10(gfo.get_sw().z)
# 
# Isolate channel cells using joint flow thresholds
domain[(Qi2d > -1.7) & (Sw2d < -1.4) & (gfo.hw.z > 0.05)] = 1
# Clean up gaps and edge noise via morphological binary closing operations
domain = binary_closing(domain, iterations=3)
# 
# Visualize the calculated binary river network mask
fig, ax = plt.subplots()
cb = ax.imshow(domain, cmap='cividis', extent=dem.extent)
ax.imshow(dem.hillshade(), cmap='gray', extent=dem.extent, alpha=0.4)
plt.show()

<div style="background-color: #121824; padding: 20px; border-radius: 8px; border-left: 6px solid #9d4edd; color: #e0e0e0;">

## 📐 Automated Channel Width Extraction

A clean, efficient post-processing workflow executed entirely via standard `scipy` and `scikit-image` sequences to map true channel width paths straight from our 2D hydraulic footprint.

<div style="padding: 10px 15px; background-color: rgba(157, 78, 221, 0.04); border-left: 4px solid #9d4edd; border-radius: 4px; font-size: 18px; margin-bottom: 15px;">
    🔗 <strong>Reference Code:</strong> Review the complete original script logic inside the official <a href="https://github.com/TopoToolbox/gallery/blob/main/notebooks/python/graphflood/graphflood.ipynb">TopoToolbox Paper Gallery Notebook</a>.
</div>

### 🔄 Processing Pipeline

<table style="width: 100%; border-collapse: collapse; margin: 15px 0; font-size: 16px;">
    <thead>
        <tr style="background-color: rgba(157, 78, 221, 0.15); border-bottom: 2px solid #9d4edd;">
            <th style="padding: 10px; text-align: left; width: 15%;">Step</th>
            <th style="padding: 10px; text-align: left; width: 30%;">Operation</th>
            <th style="padding: 10px; text-align: left; width: 55%;">Core Target Function</th>
        </tr>
    </thead>
    <tbody>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;"><strong>1</strong></td>
            <td style="padding: 12px;">Clean Channel Mask</td>
            <td style="padding: 12px;">Isolate active river zones (based on slope-discharge limits) and remove interior holes using <code>scipy.ndimage.binary_closing</code>.</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;"><strong>2</strong></td>
            <td style="padding: 12px;">Extract Centerline Spines</td>
            <td style="padding: 12px;">Thin the continuous 2D mask tracks down into an individual 1-pixel wide skeleton network via <code>skimage.morphology.skeletonize</code>.</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;"><strong>3</strong></td>
            <td style="padding: 12px;">Calculate Edge Distance</td>
            <td style="padding: 12px;">Compute exact Euclidean coordinate distances from centerline nodes to the nearest boundary edge using <code>scipy.ndimage.distance_transform_edt</code>.</td>
        </tr>
    </tbody>
</table>

</div>

In [ ]:
# =============================================================================
# 1. MORPHOLOGICAL ANALYSIS & DISTANCE TRANSFORM
# =============================================================================

# Map the exact Euclidean distance from every channel pixel to the nearest dry bank boundary
dt_edt = distance_transform_edt(domain)

# Thin the 2D binary channel mask down to a continuous, 1-pixel wide centerline network
skel = skeletonize(domain)


# =============================================================================
# 2. CHANNEL WIDTH COMPUTATION & FILTERING
# =============================================================================

# Calculate absolute physical channel width (m) along the centerline skeleton
# Formula tracks full-width (2 * half-width) and adjusts for discrete pixel resolution
width = 2 * dt_edt[skel] * dem.cellsize - dem.cellsize

# Generate a boolean filtering mask to isolate features >= 2 meters wide
mask = width >= 2


# =============================================================================
# 3. SPATIAL COORDINATE TRANSFORMATION
# =============================================================================

# Extract the 2D array matrix row and column indices for all skeleton nodes
rows_skel, cols_skel = np.where(skel)

# Project pixel index positions into true geographic map coordinates (X, Y)
xskel, yskel = ttb.transform_coords(
    dem, 
    rows_skel, 
    cols_skel,
    input_mode='indices2D', 
    output_mode='coordinates'
)


# =============================================================================
# 4. GEOGRAPHIC VISUALIZATION
# =============================================================================

fig, ax = plt.subplots()

# Base Layer: Render the topography hillshade in grayscale
ax.imshow(dem.hillshade(), cmap='gray', extent=dem.extent, vmax=1.2)
ax.imshow(domain, cmap='cividis', extent=dem.extent, alpha=0.25)

# Data Overlay: Plot the filtered centerline coordinates colored by calculated width
cb = ax.scatter(
    xskel[mask], 
    yskel[mask], 
    lw=0, 
    c=width[mask], 
    cmap='plasma'
)

# Render colorbar interface and output graphic area
plt.colorbar(cb, label='width (m)')
plt.show()

# 🏞️🏞️🏞️🏞️🏞️ Done with the Quickstart 🏞️🏞️🏞️🏞️🏞️

<div style="background-color: #121824; border: 1px solid #233047; border-radius: 8px; padding: 25px; text-align: left; margin: 20px 0;">
    <div style="font-size: 32px; letter-spacing: 15px; margin-bottom: 10px;">
        ⛰️ 🌲 🌲 🌊 🌲 ⛰️ 🌲 🌲 🌲 ⛰️ 🌲 🌲 🌊 🌲 ⛰️ 
    </div>
</div>

<div style="background-color: #121824; padding: 20px; border-radius: 8px; border-left: 6px solid #ffb703; color: #e0e0e0;">

## 🏁 Checking If the Model Is Done (Convergence)

How can we verify when the simulation has finished its job? We track our system metrics to find where local water levels stop changing and lock down into a stable state.

<table style="width: 100%; border-collapse: collapse; margin: 15px 0; font-size: 20px;">
    <thead>
        <tr style="background-color: rgba(255, 183, 3, 0.15); border-bottom: 2px solid #ffb703;">
            <th style="padding: 10px; text-align: left; width: 35%;">What We Check</th>
            <th style="padding: 10px; text-align: left; width: 30%;">Ideal Goal</th>
            <th style="padding: 10px; text-align: left; width: 35%;">What Actually Happens</th>
        </tr>
    </thead>
    <tbody>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;">📊 <strong>Water Balance</strong></td>
            <td style="padding: 12px;">Water In = Water Out<br>($Q_{in} = Q_{out}$)</td>
            <td style="padding: 12px; color: #a0a0a0;">The total volume exiting map parameters will slowly approach absolute parity with inflow records over time.</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;">🌊 <strong>Stable Water Depth</strong></td>
            <td style="padding: 12px;">Depth stops changing<br>($\frac{dh}{dt} = 0$)</td>
            <td style="padding: 12px; color: #a0a0a0;">While water depths rarely freeze completely to absolute zero change, grid values level off enough to stop mattering very quickly.</td>
        </tr>
    </tbody>
</table>

---

### ⚙️ Quick Troubleshooting Strategy

* **Run it longer:** If water surfaces are actively moving, extend processing loops using standard calls to `gfo.run_n_iterations(...)`.
* **Best way to check:** Monitor the structural changes in depth profile maps between cycles ($dh$). Once variance adjustments shrink past a nominal target baseline, your model is balanced.

</div>

In [ ]:
# Re-instantiate model instance to benchmark step-wise iteration increments
gfo = ttb.GFObject(dem, p=100e-3/3600, manning=0.05)
dhs = [gfo.hw.z.copy()]

In [ ]:
# 
N = 10
N_it = 10
# Stepwise solver tracking loop to monitor water depth grid adjustments over execution
for i in range(N):
    print(f'running {(i+1) * N_it}/{N * N_it}', end='\r', flush=True)
    gfo.run_n_iterations(dt=5e-2, n_iterations=N_it)
    # Enforce boundary zero-depth conditions at perimeter edges
    gfo.hw[[0, -1], :] = 0
    gfo.hw[:, [-1, 0]] = 0
    dhs.append(gfo.hw.z.copy())
dhs_arr = np.array(dhs)

In [ ]:
# Close any lingering plots and check depth distribution snapshots at steps 1, 2, 5, 10
# plt.close('all')
fig, axes = plt.subplots(2, 2)
cb = axes[0, 0].imshow(dhs_arr[1], cmap='Blues', extent=dem.extent, vmax=0.5)
cb = axes[0, 1].imshow(dhs_arr[2], cmap='Blues', extent=dem.extent, vmax=0.5)
cb = axes[1, 0].imshow(dhs_arr[5], cmap='Blues', extent=dem.extent, vmax=0.5)
cb = axes[1, 1].imshow(dhs_arr[10], cmap='Blues', extent=dem.extent, vmax=0.5)
plt.show()

In [ ]:
# Plot the log-scaled progression of the 90th percentile depth variance delta to assess convergence
# plt.close('all')
fig, ax = plt.subplots()
ax.plot(np.arange(len(dhs)-1)*N_it, np.percentile(np.abs(np.diff(dhs_arr, axis=0)), 90, axis=(1, 2)), lw=3, color='w')
ax.set_xlabel('N iterations')
ax.set_ylabel(f'90th perc. of h increment every {N_it} iterations')
ax.set_yscale('log')
time.sleep(0.2) # Mitigation step preventing rendering race condition bugs inside JupyterLab interface
plt.show()

<div style="background-color: #121824; padding: 20px; border-radius: 8px; border-left: 6px solid #3a86ff; color: #e0e0e0;">

## 🗺️ Boundary Conditions: The "Reach" Case

Isolate explicit river paths by defining routing directions: driving inbound stream channels through a selected boundary zone (South) and establishing exit points along another (North).

### 🔢 Boundary Condition (BC) Grid Codes

Pixel actions are set using an integer array layer that exactly mirrors our topography layout geometry:

<table style="width: 100%; border-collapse: collapse; margin: 15px 0; font-size: 16px;">
    <thead>
        <tr style="background-color: rgba(58, 134, 255, 0.15); border-bottom: 2px solid #3a86ff;">
            <th style="padding: 10px; text-align: left; width: 15%;">BC Code</th>
            <th style="padding: 10px; text-align: left; width: 25%;">Type</th>
            <th style="padding: 10px; text-align: left; width: 60%;">Flow Behavior</th>
        </tr>
    </thead>
    <tbody>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;"><code>0</code></td>
            <td style="padding: 12px;">🧱 <strong>No Data / Wall</strong></td>
            <td style="padding: 12px; color: #a0a0a0;">Solid impermeable boundary. Flow is completely blocked and cannot pass or exit.</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;"><code>1</code></td>
            <td style="padding: 12px;">🌊 <strong>Normal Active Area</strong></td>
            <td style="padding: 12px; color: #a0a0a0;">Standard interior routing. Flow moves continuously across cells but cannot scale over borders.</td>
        </tr>
        <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
            <td style="padding: 12px;"><code>3</code></td>
            <td style="padding: 12px;">🚪 <strong>Outlet Gate</strong></td>
            <td style="padding: 12px; color: #e0e0e0; font-weight: bold;">Unconstrained drainage point. Flow permanently empties out of the simulation at this node.</td>
        </tr>
    </tbody>
</table>

<div style="padding: 10px 15px; background-color: rgba(255, 0, 127, 0.08); border-left: 4px solid #ff007f; border-radius: 4px; font-size: 15px; margin-bottom: 20px;">
    ⚠️ <strong>Critical Rule:</strong> Active simulation runs require at least one pixel configured as an outlet gate (<code>3</code>). If an exit path is missing, water pools infinitely and the solver cannot balance metrics.
</div>

### 📍 Spatial Input Mapping (Point Source Discharge)

Instead of applying uniform rainfall across the entire map footprint, we switch to a **spatially variable input array**. This zeroes out fluid additions everywhere except at our designated channel source node, precisely replicating a localized upstream stream inflow.

</div>

In [ ]:
# Set default processing mask states to internal routing (1) 
bcs = np.ones_like(dem.z, dtype=np.uint8)
# Impose solid impermeable boundaries (0) around the immediate exterior rows and columns
bcs[[-1, 0], :] = 0
bcs[:, [-1, 0]] = 0

# Construct empty array map to manage explicit flow source locations
prec_in = np.zeros_like(dem.z, dtype=np.float32)

# Draw background reference surface map
fig, ax = plt.subplots()
ax.imshow(dem.hillshade(), cmap='gray')
plt.show()

In [ ]:
# Create an unconstrained drainage exit point (3) on a segment of the boundary wall
bcs[20:50, -1] = 3

# Inject a localized discharge point source of 5 m3/s scaled to a precipitation equivalent flux
prec_in[-2, 30] = 5 / (dem.cellsize**2)

In [ ]:

# Draw background reference surface map
fig, ax = plt.subplots()
ax.imshow(dem.hillshade(), cmap='gray')
ax.imshow(bcs, cmap='jet', alpha = 0.7)
plt.show()

In [ ]:
# Initialize a new Graphflood model configuration using the boundary matrix and localized flow input
gfo = ttb.GFObject(dem, p=prec_in, manning=0.05, bcs=bcs)

In [ ]:
# Execute processing loop and mask out non-contributing dry terrain nodes
gfo.run_n_iterations(dt=1e-2, n_iterations=100)
gfo.hw.z[gfo.get_qvol_i().z == 0] = 0.

In [ ]:
# View inundation track profile resulting from the isolated reach boundary settings
fig, ax = plt.subplots()
cb = ax.imshow(gfo.hw, cmap='Blues', extent=dem.extent, vmax=0.5)
ax.imshow(dem.hillshade(), cmap='gray', extent=dem.extent, alpha=0.4)
plt.colorbar(cb, label='Flow depth (m)')
plt.show()

In [ ]:
# Archive current water depth output, scale point discharge by 10x, and update flow fields
thw = gfo.hw.z.copy()
gfo.precipitations = prec_in * 10
gfo.run_n_iterations(dt=1e-3, n_iterations=100)
gfo.hw.z[gfo.get_qvol_i().z == 0] = 0.

In [ ]:
# Compare flow depth signatures between the 5 m3/s and 50 m3/s discharge inputs side-by-side
fig, axes = plt.subplots(1, 2)
axes[0].imshow(gfo.hw, cmap='Blues', extent=dem.extent, vmax=0.5)
axes[1].imshow(thw, cmap='Blues', extent=dem.extent, vmax=0.5)
axes[0].imshow(dem.hillshade(), cmap='gray', extent=dem.extent, alpha=0.4)
axes[1].imshow(dem.hillshade(), cmap='gray', extent=dem.extent, alpha=0.4)

axes[0].set_title(r'50 $m^3.s^{-1}$')
axes[1].set_title(r'5 $m^3.s^{-1}$')
plt.show()